# Ablações R2.2 / R2.3 no Colab: todos os experimentos que faltam

Este notebook completa a grade de `run_experiments_ablation.py`: 63 experimentos (3 modelos × 7 variantes × 3 datasets), com profundidade e receptive field. A profundidade roda primeiro.

**O que já rodou não é executado de novo.** Um experimento é pulado quando o JSON dele existe em `ifd_ablation/results_ablation` no Drive. Os 4 resultados da CNN no PU que estão no repositório são copiados para lá automaticamente, então faltam 59.

**Estimativa:** cerca de 410 unidades de computação (CU) e 75 h de A100 40 GB, ou até 585 CU com folga. São estimativas, porque nada disso foi medido ainda no Colab.

## Antes de começar: resultados fora do repositório

Se você tem outros resultados que não estão no GitHub, por exemplo a profundidade da CNN nos CWRU, envie os JSONs para o Drive antes do Passo 2. Mantenha a pasta do dataset e o nome do arquivo:

```
ifd_ablation/results_ablation/CWRU12k/cnn1d_ablation_d1_k3_cwru12k.json
```

A célula 6 mostra quantos experimentos foram reconhecidos como concluídos. Confira esse número antes de deixar rodando.

## Passo 1: preparar os dados (uma vez só, em CPU)

Baixar e converter os dados em CPU evita pagar a A100 com a GPU parada.

1. Em **Ambiente de execução > Alterar o tipo de ambiente de execução**, escolha **CPU**.
2. Clique em **Ambiente de execução > Executar tudo** e autorize o acesso ao Google Drive.
3. Espere o erro `Dados prontos` na célula 5. Esse erro é esperado: ele para o notebook antes do treino.
4. Confira se `ifd_ablation/dados_ablacao.tar` apareceu no seu Drive (precisa de cerca de 7 GB livres).
5. Clique em **Ambiente de execução > Desconectar e excluir ambiente de execução**.

## Passo 2: rodar os experimentos (em A100)

1. Troque o ambiente para **GPU A100**. Se você tem Colab Pro+, ligue **Execução em segundo plano** na mesma janela.
2. Clique em **Executar tudo**.
3. Fique por perto nos primeiros minutos. A célula 7 testa a ResNet-34 no PU, que é o caso de maior memória, e só roda uma vez.
4. Quando a célula 8 começar a treinar, pode deixar rodando.

Ao terminar, a célula 8 grava tudo no Drive e **desconecta a máquina sozinha** para parar o consumo de CU.

## Passo 3: quando a sessão cair

O Colab encerra a sessão depois de no máximo 24 h, então espere umas 4 sessões. A cada queda:

1. Reconecte com a mesma A100 e clique em **Executar tudo**.
2. Repita até a célula 6 mostrar `Pendentes: 0`.

Só o experimento que estava em andamento é perdido, porque cada resultado é salvo no Drive assim que termina.

## Como manter a sessão ativa

- **Com Colab Pro+ e execução em segundo plano:** pode fechar a aba e desligar o computador. A sessão segue por até 24 h enquanto houver CU.
- **Sem execução em segundo plano (Pro ou pay-as-you-go):** deixe a aba aberta e o computador acordado (no Mac, rode `caffeinate -dims` no Terminal). Interaja com a página de vez em quando, por exemplo clicando em uma célula ou rolando. O Google não publica o limite de inatividade; relatos falam em cerca de 90 minutos.
- **Em qualquer plano:** interagir evita o corte por inatividade, mas não aumenta o limite máximo da sessão. Evite scripts de clique automático, que tentam burlar esse limite.

## Depois de terminar

1. Baixe a pasta `ifd_ablation/results_ablation` do Drive.
2. Copie o conteúdo para `results_ablation/` no repositório local.
3. Rode `python run_experiments_ablation.py --report`. A célula 9 também gera o relatório no Drive.

## Problemas comuns

| Sintoma | O que fazer |
|---|---|
| Célula 7: `CUDA out of memory` | A ResNet-34 não coube. Mude `PULAR_TESTE_MEMORIA = True` na célula 1 e rode de novo. Só `d34_k7` no PU vai falhar; depois rode esse experimento em uma A100 80 GB. |
| Célula 5: `GPU não é A100` | O Colab entregou outra GPU. Reconecte ou mude `PERMITIR_OUTRA_GPU = True`, sabendo que a ResNet no PU pode não caber. |
| Célula 4: erro no download | O Google Drive limita downloads dos arquivos públicos. Espere algumas horas e repita o Passo 1. |
| Célula 4: pede o Passo 1 | O arquivo de dados não está no Drive. Faça o Passo 1 em CPU. |
| CU acabando | A profundidade roda primeiro. Pare quando quiser: o que terminou está salvo, e a célula 8 retoma depois do ponto em que parou. |

In [ ]:
# 1. Configuração. Ajuste aqui se precisar; o restante do notebook não precisa de edição.
import glob, json, os, shutil, subprocess, sys, tarfile, time

REPO_URL = "https://github.com/falconeee/unbiased-ifd-benchmark.git"
BRANCH = "main"
REPO_DIR = "/content/unbiased-ifd-benchmark"
DATA_ROOT = "/content/data"                # dados na VM: leitura rápida, apagados a cada sessão
DRIVE_ROOT = "/content/drive"
DRIVE_DIR = f"{DRIVE_ROOT}/MyDrive/ifd_ablation"
RESULTS_DIR = f"{DRIVE_DIR}/results_ablation"
DATA_ARCHIVE = f"{DRIVE_DIR}/dados_ablacao.tar"
SMOKE_MARKER = f"{DRIVE_DIR}/teste_memoria_ok.txt"

DATASETS = ["PU", "CWRU12k", "CWRU48k"]
DEVICE = "cuda"

# Ordem de execução. Cada etapa pula o que já tem resultado no Drive.
ETAPAS = [
    ["--study", "depth"],                  # prioridade: profundidade dos três modelos
    ["--all"],                             # o restante: receptive field
]
ARGS_EXTRAS = []                           # só para testes; os experimentos reais não usam

# Caso de maior memória: ResNet-34 no PU com batch 128 (cerca de 33 GB).
TESTE_MEMORIA = ["--model", "resnet18", "--variant", "d34_k7", "--dataset", "PU",
                 "--epochs", "2", "--max-rounds", "1"]
PULAR_TESTE_MEMORIA = False
PERMITIR_OUTRA_GPU = False
EXIGIR_GPU = True
PREPARAR_DADOS_EM_GPU = False              # True gasta CU da GPU no download e na conversão
SALVAR_ARTEFATOS = False                   # checkpoints somariam mais de 20 GB no Drive
DESCONECTAR_AO_TERMINAR = True

In [ ]:
# 2. Drive e funções auxiliares.
from google.colab import drive

drive.mount(DRIVE_ROOT)
os.makedirs(RESULTS_DIR, exist_ok=True)

VENV_PY = f"{REPO_DIR}/.venv/bin/python"
ENV = dict(os.environ, MPLBACKEND="Agg", PYTHONUNBUFFERED="1")
ENV.pop("PYTHONPATH", None)                # o PYTHONPATH do Colab não deve vazar para o .venv


def rodar(cmd, cwd=None, check=True):
    # Executa um comando mostrando a saída linha a linha.
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=cwd, env=ENV, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    linhas = []
    for linha in proc.stdout:
        print(linha, end="", flush=True)
        linhas.append(linha)
    proc.wait()
    if check and proc.returncode != 0:
        raise RuntimeError(f"Comando falhou (código {proc.returncode}): {' '.join(cmd)}")
    return proc.returncode, "".join(linhas)


def py(codigo):
    # Roda um trecho de Python no ambiente do projeto e devolve o que ele imprimiu.
    return subprocess.run([VENV_PY, "-c", codigo], cwd=REPO_DIR, env=ENV, text=True,
                          capture_output=True, check=True).stdout


def ablacao(*args):
    # Chama o runner das ablações com os caminhos deste notebook.
    extra = [] if SALVAR_ARTEFATOS else ["--no-artifacts"]
    return [VENV_PY, "run_experiments_ablation.py", *args, "--device", DEVICE, "--no-download",
            "--data-root", DATA_ROOT, *extra]

In [ ]:
# 3. Código e dependências, nas versões exatas do uv.lock.
if not os.path.isdir(f"{REPO_DIR}/.git"):
    rodar(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR])
print("commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR,
                                         text=True).strip())

if shutil.which("uv") is None:
    rodar([sys.executable, "-m", "pip", "install", "-q", "uv"])
rodar(["uv", "sync", "--frozen", "--no-dev"], cwd=REPO_DIR)
print(py("import sys, torch; print('python', sys.version.split()[0], '| torch', torch.__version__)"))

In [ ]:
# 4. Dados: restaura do Drive ou, na primeira vez, baixa, converte e guarda no Drive.
PREPARAR = '''
import sys
from src import folds
from src.datasets import DATASETS, load_deep_dataset
for nome in sys.argv[2:]:
    print("preparando", nome, flush=True)
    spec = DATASETS[nome]
    deep = load_deep_dataset(spec, sys.argv[1], download=True)
    folds.build_folds(spec, deep, "multi_round", sys.argv[1], "notebook")
'''


def guardar_dados():
    # Guarda os dados convertidos. Do raw_data só vão as pastas: o vibdata checa que elas
    # existem, mas não relê os arquivos brutos quando o cache convertido é válido.
    local = "/content/dados_ablacao.tar"
    with tarfile.open(local, "w") as tar:
        for raiz, _, _ in os.walk(f"{DATA_ROOT}/raw_data"):
            tar.add(raiz, arcname=os.path.relpath(raiz, DATA_ROOT), recursive=False)
        for pasta in ("deep_data", "grouping"):
            tar.add(f"{DATA_ROOT}/{pasta}", arcname=pasta)
    shutil.copyfile(local, DATA_ARCHIVE + ".parcial")
    os.replace(DATA_ARCHIVE + ".parcial", DATA_ARCHIVE)   # só aparece no Drive se estiver completo
    os.remove(local)


if os.path.isdir(f"{DATA_ROOT}/deep_data"):
    print("Dados já estão nesta sessão.")
elif os.path.exists(DATA_ARCHIVE):
    print("Restaurando dados do Drive...", flush=True)
    os.makedirs(DATA_ROOT, exist_ok=True)
    with tarfile.open(DATA_ARCHIVE) as tar:
        tar.extractall(DATA_ROOT, filter="data")
    print("Dados restaurados.")
else:
    if shutil.which("nvidia-smi") and not PREPARAR_DADOS_EM_GPU:
        raise RuntimeError("Dados ainda não preparados. Faça o Passo 1 em um ambiente de CPU "
                           "para não gastar CU da GPU.")
    rodar([VENV_PY, "-c", PREPARAR, DATA_ROOT, *DATASETS], cwd=REPO_DIR)
    guardar_dados()
    print(f"Dados guardados em {DATA_ARCHIVE} "
          f"({os.path.getsize(DATA_ARCHIVE) / 1e9:.1f} GB).")

In [ ]:
# 5. Confere a GPU. Em ambiente de CPU, o notebook para aqui de propósito.
if EXIGIR_GPU:
    if shutil.which("nvidia-smi") is None:
        raise RuntimeError("Dados prontos. Agora siga o Passo 2: troque o ambiente para A100 "
                           "e clique em Executar tudo.")

    CHECAR = ("import torch; print(torch.cuda.is_available()); "
              "print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else ''); "
              "print(torch.cuda.get_device_properties(0).total_memory / 2**30 "
              "if torch.cuda.is_available() else 0)")
    ok, nome, memoria = py(CHECAR).split("\n")[:3]
    if ok != "True":
        # O uv.lock usa torch para CUDA 13. Se o driver for mais antigo, troca pela mesma
        # versão do torch compilada para CUDA 12.6.
        print("torch não enxergou a GPU; instalando a build CUDA 12.6.", flush=True)
        versao = py("import torch; print(torch.__version__.split('+')[0])").strip()
        rodar(["uv", "pip", "install", "--python", VENV_PY, "--reinstall-package", "torch",
               f"torch=={versao}+cu126",
               "--index", "https://download.pytorch.org/whl/cu126"], cwd=REPO_DIR)
        ok, nome, memoria = py(CHECAR).split("\n")[:3]
        if ok != "True":
            raise RuntimeError("O PyTorch não enxergou a GPU. Veja a saída de nvidia-smi.")

    print(f"GPU: {nome} ({float(memoria):.0f} GB)")
    if "A100" not in nome and not PERMITIR_OUTRA_GPU:
        raise RuntimeError(f"GPU não é A100: {nome}. Reconecte ou use PERMITIR_OUTRA_GPU = True.")

In [ ]:
# 6. Situação: traz os resultados já versionados, confere o Drive e conta o que falta.
def situacao():
    specs = json.loads(py(
        "import json; from ablation.registry import REGISTRY as R; "
        "print(json.dumps([[s.dataset, s.family, s.key, s.slug] for s in R]))"))
    faltam = [s for s in specs
              if not os.path.exists(f"{RESULTS_DIR}/{s[0]}/{s[3]}.json")]
    print(f"Concluídos: {len(specs) - len(faltam)} de {len(specs)} | Pendentes: {len(faltam)}")
    resumo = {}
    for dataset, familia, _, _ in faltam:
        resumo[(dataset, familia)] = resumo.get((dataset, familia), 0) + 1
    for (dataset, familia), total in sorted(resumo.items()):
        print(f"  {dataset:<9}{familia:<10}{total}")
    return faltam


# Os resultados que já estão no repositório não devem ser refeitos.
for origem in glob.glob(f"{REPO_DIR}/results_ablation/*/*.json"):
    destino = origem.replace(f"{REPO_DIR}/results_ablation", RESULTS_DIR)
    if not os.path.exists(destino):
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        shutil.copyfile(origem, destino)

# Um JSON cortado pela queda da sessão faria o --resume pular um experimento incompleto.
for caminho in glob.glob(f"{RESULTS_DIR}/*/*.json"):
    if os.path.basename(caminho).startswith("_"):
        continue
    try:
        with open(caminho, encoding="utf-8") as arquivo:
            json.load(arquivo)
    except ValueError:
        os.replace(caminho, caminho + ".corrompido")
        print("JSON incompleto separado; o experimento será refeito:", caminho)

pendentes = situacao()

In [ ]:
# 7. Teste de memória: 2 épocas da ResNet-34 no PU, antes de gastar o saldo. Roda uma vez só.
if PULAR_TESTE_MEMORIA:
    print("Teste de memória pulado.")
elif os.path.exists(SMOKE_MARKER):
    print("Teste de memória já aprovado em uma sessão anterior.")
elif not pendentes:
    print("Nada pendente; teste desnecessário.")
else:
    codigo, saida = rodar(ablacao(*TESTE_MEMORIA, *ARGS_EXTRAS, "--output-dir", "/content/teste"),
                          cwd=REPO_DIR, check=False)
    if codigo != 0:
        if "out of memory" in saida.lower():
            raise RuntimeError("A ResNet-34 não coube na GPU. Veja Problemas comuns no início.")
        raise RuntimeError("O teste falhou; veja o erro acima antes de gastar CU.")
    with open(SMOKE_MARKER, "w", encoding="utf-8") as marcador:
        marcador.write(time.strftime("%Y-%m-%d %H:%M:%S\n"))
    print("Teste aprovado. Confira o tempo por época acima; a estimativa é de cerca de 30 s.")

In [ ]:
# 8. Execução principal. Pode deixar rodando; ao terminar, a máquina desconecta sozinha.
inicio = time.time()
for etapa in ETAPAS:
    rodar(ablacao(*etapa, *ARGS_EXTRAS, "--resume", "--keep-going", "--output-dir", RESULTS_DIR),
          cwd=REPO_DIR, check=False)

print(f"\nTempo nesta sessão: {(time.time() - inicio) / 3600:.1f} h")
pendentes = situacao()
print("Se restou algo pendente, veja as falhas em", f"{RESULTS_DIR}/run.log")

if DESCONECTAR_AO_TERMINAR:
    drive.flush_and_unmount()              # garante que tudo chegou ao Drive antes de desligar
    from google.colab import runtime

    runtime.unassign()

## Relatório

A célula 8 já gera o relatório ao terminar. Use a célula abaixo só para ver resultados parciais. Ela não roda em **Executar tudo** quando a célula 8 desconecta a máquina.

In [ ]:
# 9. Relatório parcial, sob demanda. Funciona em CPU depois de rodar as células 1 a 3.
rodar([VENV_PY, "run_experiments_ablation.py", "--report", "--output-dir", RESULTS_DIR],
      cwd=REPO_DIR)
print("Relatório em", f"{RESULTS_DIR}/report.md")